# Taco Alley complaint structuring with supervised fine-tuning

This supervised fine-tuning (SFT) workflow updates all weights of Qwen3-1.7B to convert incoming complaints with fields `customer_id`, `date`, `location`, and `message` into processed complaint JSON by adding `category`, `sub_category`, and `tone_urgency`.

## Hardware guidance

These settings are a starting point, not a guaranteed fit in 16 GB VRAM. Memory and runtime depend on the model, precision, optimizer, batch size, sequence length, and evaluation workload. Reduce the batch size or use a smaller model when needed. Validate a short run on your selected GPU before committing to a full training run.

Low-rank adaptation (LoRA) and quantized LoRA (QLoRA) can reduce memory requirements. See the [LoRA SFT notebook](sft_lora_run.ipynb) and [QLoRA SFT notebook](sft_qlora_run.ipynb) for those workflows.

<a href="https://colab.research.google.com/github/ned1313/Fine-tuning-and-Optimizing-Small-Language-Models/blob/main/notebooks/sft_full_run.ipynb" target="_parent">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

## Google Colab prep

If you are running this notebook in Google Colab, select a GPU runtime and run the code block below to install the necessary packages. If you are running locally with the project dependencies already installed, skip this cell.

The current code uses BF16 on the GPU. Select a BF16-capable runtime, such as a supported Ampere-or-newer Nvidia GPU, or explicitly adapt and test the precision settings before using a T4. The `SFTConfig` precision flags currently use `torch.dtype` objects rather than Boolean values and need a separate code correction; selecting a compatible GPU alone does not resolve that configuration issue.

The install cell installs packages only. Clone or upload the repository into the Colab runtime, including its datasets, and set the kernel's working directory to the repository's `notebooks` directory before continuing. This also keeps relative data, credential, and artifact paths consistent with local runs. Preserve any generated artifacts you need before the runtime is discarded.

For local runs, also set the kernel's working directory to the repository's `notebooks` directory. Opening the notebook in an editor does not by itself guarantee that working directory.

In [8]:
%pip install "transformers>=4.46.0" "trl>=0.11.0" "datasets>=2.21.0" "accelerate>=0.34.0" "sentencepiece>=0.2.0" "matplotlib>=3.8.0"

## Demonstration: Preparing datasets as training input

Load `datasets/taco_alley_customer_complaints.csv`, validate the required complaint schema, and build train/eval splits for supervised fine-tuning.

In [11]:
import csv
import json
import math
import random
import statistics
import time
from collections import Counter
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import matplotlib.pyplot as plt
import torch
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

# Reproducibility for split and sampling.
SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

# 16GB-friendly default base model for full fine-tuning.
BASE_MODEL = "Qwen/Qwen3-1.7B"
MAX_SEQ_LENGTH = 384
TRAIN_EVAL_SPLIT = 0.15

# Incoming complaints arrive with these fields.
INPUT_FIELDS = ["customer_id", "date", "location", "message"]
# Processed complaints add these fields.
OUTPUT_FIELDS = ["category", "sub_category", "tone_urgency"]
REQUIRED_FIELDS = [*INPUT_FIELDS, *OUTPUT_FIELDS]

EVAL_SAMPLE_SIZE = 120


def resolve_project_root(start: Path) -> Path:
    current = start.resolve()
    for parent in [current, *current.parents]:
        if (parent / "pyproject.toml").exists():
            return parent
    return current


PROJECT_ROOT = resolve_project_root(Path.cwd())
DATASETS_DIR = PROJECT_ROOT / "datasets"
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"
TRAINING_DATA_PATH = DATASETS_DIR / "taco_alley_customer_complaints.csv"
if not TRAINING_DATA_PATH.exists():
    raise FileNotFoundError(f"Training dataset not found at: {TRAINING_DATA_PATH}")

output_dir = ARTIFACTS_DIR / "sft_full_run"
output_dir.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Training data source: {TRAINING_DATA_PATH}")

Project root: /content
Training data source: /content/datasets/taco_alley_customer_complaints.csv


In [12]:
import re

with TRAINING_DATA_PATH.open("r", encoding="utf-8", newline="") as f:
    raw_rows = list(csv.DictReader(f))

all_columns = list(raw_rows[0].keys())
missing_columns = [c for c in REQUIRED_FIELDS if c not in all_columns]
if missing_columns:
    raise ValueError(f"Dataset is missing required columns: {missing_columns}")

tone_allowed = {"low", "moderate", "high", "urgent"}
clean_examples = []
dropped_rows = 0

for row in raw_rows:
    record = {k: str(row.get(k, "")).strip() for k in REQUIRED_FIELDS}
    # Drop any rows that are missing required fields
    if any(not record[k] for k in REQUIRED_FIELDS):
        dropped_rows += 1
        continue
    # Drop rows with invalid customer_id matching CUST-{5digit number}
    if not re.match(r"^CUST-\d{5,}",record["customer_id"]):
        dropped_rows += 1
        continue
    # Drop rows with invalid tone_urgency values
    if record["tone_urgency"].lower() not in tone_allowed:
        dropped_rows += 1
        continue
    # Drop rows with invalid dates or dates in the future
    if not re.match(r"^\d{4}-\d{2}-\d{2}$", record["date"]):
        dropped_rows += 1
        continue
    if record["date"] > time.strftime("%Y-%m-%d"):
        dropped_rows += 1
        continue

    input_obj = {k: record[k] for k in INPUT_FIELDS}
    target_obj = {k: record[k] for k in REQUIRED_FIELDS}
    clean_examples.append(
        {
            "input_obj": input_obj,
            "target_obj": target_obj,
            "target_json": json.dumps(target_obj, ensure_ascii=False),
        }
    )

dataset = Dataset.from_list(clean_examples).train_test_split(test_size=TRAIN_EVAL_SPLIT, seed=SEED)
train_ds = dataset["train"]
eval_ds = dataset["test"]

category_counts = Counter(x["target_obj"]["category"] for x in clean_examples)
tone_counts = Counter(x["target_obj"]["tone_urgency"] for x in clean_examples)

print(f"Loaded rows: {len(raw_rows)}")
print(f"Rows kept: {len(clean_examples)}")
print(f"Rows dropped: {dropped_rows}")
print(f"Train size: {len(train_ds)}")
print(f"Eval size: {len(eval_ds)}")
print("Categories:", dict(category_counts))
print("Tone distribution:", dict(tone_counts))

Loaded rows: 1000
Rows kept: 971
Rows dropped: 29
Train size: 825
Eval size: 146
Categories: {'Food Quality': 221, 'Cleanliness': 79, 'App & Online Ordering': 85, 'Order Accuracy': 193, 'Delivery': 118, 'Billing & Pricing': 99, 'Service': 111, 'Product Availability': 65}
Tone distribution: {'Low': 312, 'High': 202, 'Moderate': 402, 'Urgent': 55}


In [13]:
train_ds[0]

{'input_obj': {'customer_id': 'CUST-49229',
  'date': '2026-02-01',
  'location': 'King of Prussia, PA',
  'message': 'The payment failed in the app, but I was still charged twice.'},
 'target_obj': {'category': 'Billing & Pricing',
  'customer_id': 'CUST-49229',
  'date': '2026-02-01',
  'location': 'King of Prussia, PA',
  'message': 'The payment failed in the app, but I was still charged twice.',
  'sub_category': 'Duplicate charge',
  'tone_urgency': 'Low'},
 'target_json': '{"customer_id": "CUST-49229", "date": "2026-02-01", "location": "King of Prussia, PA", "message": "The payment failed in the app, but I was still charged twice.", "category": "Billing & Pricing", "sub_category": "Duplicate charge", "tone_urgency": "Low"}'}

## Demonstration: Tokenizers and chat templates for structured complaints

Convert each complaint pair into a chat-style training sample with a strict JSON target and inspect tokenization behavior before training.

In [14]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"Tokenizer: {tokenizer.__class__.__name__}")
print(f"Tokenizer pad token: {tokenizer.pad_token}")
print(f"Tokenizer eos token: {tokenizer.eos_token}")

config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

Tokenizer: Qwen2Tokenizer
Tokenizer pad token: <|endoftext|>
Tokenizer eos token: <|im_end|>


In [15]:

SYSTEM_PROMPT = (
    "You are a complaint structuring assistant for Taco Alley. "
    "Input is a JSON object with fields customer_id, date, location, message. "
    "Return only valid JSON with exactly these fields in the output: "
    f"{REQUIRED_FIELDS}. "
    "Keep the original input fields unchanged and add correct values for category, sub_category, and tone_urgency. "
    "Do not add any extra keys or commentary."
)

def format_split_for_sft(split_ds: Dataset) -> Dataset:
    formatted_rows = []
    for ex in split_ds:
        train_messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": json.dumps(ex["input_obj"], ensure_ascii=False)},
            {"role": "assistant", "content": ex["target_json"]},
        ]
        infer_messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": json.dumps(ex["input_obj"], ensure_ascii=False)},
        ]

        try:
            text = tokenizer.apply_chat_template(
                train_messages, tokenize=False, add_generation_prompt=False, enable_thinking=False
            )
            prompt_text = tokenizer.apply_chat_template(
                infer_messages, tokenize=False, add_generation_prompt=True, enable_thinking=False
            )
        except Exception:
            text = (
                f"SYSTEM: {SYSTEM_PROMPT}\n\n"
                f"USER: {json.dumps(ex['input_obj'], ensure_ascii=False)}\n\n"
                f"ASSISTANT: {ex['target_json']}"
            )
            prompt_text = (
                f"SYSTEM: {SYSTEM_PROMPT}\n\n"
                f"USER: {json.dumps(ex['input_obj'], ensure_ascii=False)}\n\n"
                "ASSISTANT:"
            )

        formatted_rows.append(
            {
                "input_obj": ex["input_obj"],
                "target_obj": ex["target_obj"],
                "target_json": ex["target_json"],
                "text": text,
                "prompt_text": prompt_text,
            }
        )
    return Dataset.from_list(formatted_rows)

train_ds = format_split_for_sft(train_ds)
eval_ds = format_split_for_sft(eval_ds)

train_lengths = [len(tokenizer(x["text"]).input_ids) for x in train_ds]
print(f"Sample train token length mean: {statistics.mean(train_lengths):.1f}")
print(f"Sample train token length p95: {sorted(train_lengths)[int(0.95 * (len(train_lengths)-1))]}")
print(f"Sample train token length max: {max(train_lengths)}")
print("\nFormatted sample:\n")
print(train_ds[0]["text"][:1200])

Sample train token length mean: 259.2
Sample train token length p95: 282
Sample train token length max: 293

Formatted sample:

<|im_start|>system
You are a complaint structuring assistant for Taco Alley. Input is a JSON object with fields customer_id, date, location, message. Return only valid JSON with exactly these fields in the output: ['customer_id', 'date', 'location', 'message', 'category', 'sub_category', 'tone_urgency']. Keep the original input fields unchanged and add correct values for category, sub_category, and tone_urgency. Do not add any extra keys or commentary.<|im_end|>
<|im_start|>user
{"customer_id": "CUST-49229", "date": "2026-02-01", "location": "King of Prussia, PA", "message": "The payment failed in the app, but I was still charged twice."}<|im_end|>
<|im_start|>assistant
<think>

</think>

{"customer_id": "CUST-49229", "date": "2026-02-01", "location": "King of Prussia, PA", "message": "The payment failed in the app, but I was still charged twice.", "category":

## Demonstration: Configuring SFT training properties with TRL

Build `SFTConfig` and `SFTTrainer` for a full fine-tuning run. Review the hardware and precision guidance above before training; these settings do not guarantee a fit in 16 GB VRAM.

In [16]:
from trl import SFTConfig
from trl import SFTTrainer

# Check to see if this is an AMD card
device_map = "auto" if torch.cuda.is_available() else None
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    use_safetensors=True,
    device_map=device_map,
)

model.config.use_cache = False  # Better memory behavior with gradient checkpointing.
model.gradient_checkpointing_enable()


model.safetensors.index.json:   0%|          | 0.00/25.6k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

In [17]:

sft_config = SFTConfig(
    max_length=MAX_SEQ_LENGTH,
    num_train_epochs=1,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=10,
    seed=SEED,
    eval_strategy="steps",
    eval_steps=5,
    save_strategy="steps",
    save_steps=5,
    output_dir=str(output_dir / "checkpoints"),
    learning_rate=2e-5,
    lr_scheduler_type="cosine",
    warmup_steps=0.03,
    max_grad_norm=1.0,
    weight_decay=0.01,
    gradient_checkpointing=True,
    bf16=torch.bfloat16 if torch.cuda.is_available() else False,
    fp16=torch.float16 if not torch.cuda.is_available() else False,
    logging_steps=1,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    processing_class=tokenizer,
)


Adding EOS to train dataset:   0%|          | 0/825 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/825 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/825 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/825 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/825 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/146 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/146 [00:00<?, ? examples/s]

Building labels for eval dataset:   0%|          | 0/146 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/146 [00:00<?, ? examples/s]

Dropping fully masked examples from eval dataset:   0%|          | 0/146 [00:00<?, ? examples/s]

## Demonstration: Evaluate the baseline

Define the structured-output evaluation helpers and measure the base model before training. Later sections train the model, inspect metrics, save the full checkpoint, and compare post-training results with this baseline.

In [18]:
def parse_json_from_output(text: str) -> Dict[str, Any]:
    cleaned = text.strip()
    if cleaned.startswith("```"):
        lines = [line for line in cleaned.splitlines() if not line.strip().startswith("```")]
        cleaned = "\n".join(lines).strip()
    try:
        return json.loads(cleaned)
    except Exception:
        start = cleaned.find("{")
        end = cleaned.rfind("}")
        if start == -1 or end == -1 or end <= start:
            raise ValueError("No JSON object found in model output.")
        return json.loads(cleaned[start : end + 1])

def generate_structured_prediction(
    model_obj: AutoModelForCausalLM,
    input_obj: Dict[str, str],
    max_new_tokens: int = 220,
) -> Tuple[Optional[Dict[str, Any]], str]:
    prompt_text = input_obj.get("prompt_text")
    if prompt_text is None:
        infer_messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": json.dumps(input_obj, ensure_ascii=False)},
        ]
        prompt_text = tokenizer.apply_chat_template(
            infer_messages, tokenize=False, add_generation_prompt=True
        )

    inputs = tokenizer(prompt_text, return_tensors="pt").to(model_obj.device)
    with torch.no_grad():
        output = model_obj.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    generated_ids = output[0][inputs["input_ids"].shape[1] :]
    raw_text = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()
    try:
        parsed = parse_json_from_output(raw_text)
        normalized = {k: str(parsed.get(k, "")).strip() for k in REQUIRED_FIELDS}
        return normalized, raw_text
    except Exception:
        return None, raw_text

def evaluate_model(model_obj: AutoModelForCausalLM, split_ds: Dataset, label: str) -> Dict[str, Any]:
    sample_n = min(EVAL_SAMPLE_SIZE, len(split_ds))
    subset = split_ds.select(range(sample_n))
    rows = []
    started = time.perf_counter()

    print(f"Evaluating {label} model on {sample_n} samples...")
    for i, ex in enumerate(subset, start=1):
        pred_obj, raw = generate_structured_prediction(model_obj, ex["input_obj"] )
        gold = ex["target_obj"]

        schema_valid = 1.0
        if pred_obj is None:
            schema_valid = 0.0
            category_match = 0.0
            sub_category_match = 0.0
            tone_match = 0.0
        else:
            if any(not str(pred_obj.get(k, "")).strip() for k in REQUIRED_FIELDS):
                schema_valid = 0.0
            if not str(pred_obj.get("customer_id", "")).startswith("CUST-"):
                schema_valid = 0.0
            if str(pred_obj.get("tone_urgency", "")).strip().lower() not in {"low", "moderate", "high", "urgent"}:
                schema_valid = 0.0

            category_match = 1.0 if str(pred_obj.get("category", "")).strip() == str(gold.get("category", "")).strip() else 0.0
            sub_category_match = 1.0 if str(pred_obj.get("sub_category", "")).strip() == str(gold.get("sub_category", "")).strip() else 0.0
            tone_match = 1.0 if str(pred_obj.get("tone_urgency", "")).strip() == str(gold.get("tone_urgency", "")).strip() else 0.0

        weighted_score = 0.6 * schema_valid + 0.3 * category_match + 0.1 * tone_match
        rows.append(
            {
                "schema_valid": schema_valid,
                "category_match": category_match,
                "sub_category_match": sub_category_match,
                "tone_match": tone_match,
                "weighted_score": weighted_score,
                "raw_preview": raw[:220].replace("\n", " "),
            }
        )

        if i % 25 == 0 or i == sample_n:
            print(f"  Processed {i}/{sample_n} samples")

    elapsed = time.perf_counter() - started
    summary = {
        "label": label,
        "samples": sample_n,
        "seconds": elapsed,
        "samples_per_second": sample_n / elapsed if elapsed > 0 else 0.0,
        "schema_valid_rate": statistics.mean(r["schema_valid"] for r in rows),
        "category_accuracy": statistics.mean(r["category_match"] for r in rows),
        "sub_category_accuracy": statistics.mean(r["sub_category_match"] for r in rows),
        "tone_accuracy": statistics.mean(r["tone_match"] for r in rows),
        "weighted_score": statistics.mean(r["weighted_score"] for r in rows),
        "rows": rows,
    }
    return summary

print("Running baseline evaluation (before training)...")
baseline_metrics = evaluate_model(model, eval_ds, label="baseline")

for k, v in baseline_metrics.items():
    if k != "rows":
        print(f"{k}: {v}")

Running baseline evaluation (before training)...
Evaluating baseline model on 120 samples...
  Processed 25/120 samples
  Processed 50/120 samples
  Processed 75/120 samples
  Processed 100/120 samples
  Processed 120/120 samples
label: baseline
samples: 120
seconds: 1529.76696279
samples_per_second: 0.0784433204003459
schema_valid_rate: 0.0
category_accuracy: 0.0
sub_category_accuracy: 0.0
tone_accuracy: 0.0
weighted_score: 0.0


## Demonstration: Train and inspect loss

Run full fine-tuning while measuring elapsed time and GPU memory. The following metrics cell reports training statistics and plots the loss curves.

In [19]:
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    start_mem = torch.cuda.memory_allocated() / (1024 ** 3)
else:
    start_mem = 0.0

train_start = time.perf_counter()
train_result = trainer.train()
train_seconds = time.perf_counter() - train_start

RuntimeError: gemm input type at::BFloat16 and output type float is only supported for CUDA devices with compute capability 8.0 or higher

### Save the model

The next cell reports training metrics, plots loss, and saves the complete model and tokenizer beneath `artifacts/sft_full_run/final_model`. This output is a full-model checkpoint, not an adapter-only export.

In [ ]:

train_metrics = dict(train_result.metrics)
train_metrics["train_wall_seconds"] = float(train_seconds)
if torch.cuda.is_available():
    peak_mem = torch.cuda.max_memory_allocated() / (1024 ** 3)
    end_mem = torch.cuda.memory_allocated() / (1024 ** 3)
else:
    peak_mem = 0.0
    end_mem = 0.0

print("\nTraining metrics summary:")
for k, v in train_metrics.items():
    print(f"{k}: {v}")
print(f"start_mem_gb: {start_mem:.2f}")
print(f"peak_mem_gb: {peak_mem:.2f}")
print(f"end_mem_gb: {end_mem:.2f}")

log_history = trainer.state.log_history
train_steps = [x.get("step") for x in log_history if "loss" in x and "eval_loss" not in x]
train_losses = [x.get("loss") for x in log_history if "loss" in x and "eval_loss" not in x]
eval_steps = [x.get("step") for x in log_history if "eval_loss" in x]
eval_losses = [x.get("eval_loss") for x in log_history if "eval_loss" in x]

if train_steps:
    plt.figure(figsize=(9, 5))
    plt.plot(train_steps, train_losses, label="train_loss")
    if eval_steps:
        plt.plot(eval_steps, eval_losses, label="eval_loss")
    plt.xlabel("Step")
    plt.ylabel("Loss")
    plt.title("SFT Loss Curves")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.show()
else:
    print("No log points available. Skipping loss chart.")

final_dir = output_dir / "final_model"
trainer.save_model(str(final_dir))
tokenizer.save_pretrained(str(final_dir))
print(f"Saved fine-tuned model to: {final_dir}")

## Demonstration: Reload and evaluate

Release the training resources, reload the saved full model, and evaluate it on the held-out complaints. Compare the structured-output metrics with the pre-training baseline.

In [ ]:

import gc

# Release training resources before post-training evaluation.
if torch.cuda.is_available():
    try:
      del trainer
      del model
    except Exception:
      pass
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()


In [ ]:

# Load the model back for evaluation
model = AutoModelForCausalLM.from_pretrained(
    str(final_dir),
    dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    use_safetensors=True,
    device_map=device_map,
)

post_metrics = evaluate_model(model, eval_ds, label="fine_tuned")
print("\nPost-training evaluation summary:")
for k, v in post_metrics.items():
    if k != "rows":
        print(f"{k}: {v}")

comparison = {
    "schema_valid_delta": post_metrics["schema_valid_rate"] - baseline_metrics["schema_valid_rate"],
    "category_accuracy_delta": post_metrics["category_accuracy"] - baseline_metrics["category_accuracy"],
    "sub_category_accuracy_delta": post_metrics["sub_category_accuracy"] - baseline_metrics["sub_category_accuracy"],
    "tone_accuracy_delta": post_metrics["tone_accuracy"] - baseline_metrics["tone_accuracy"],
    "weighted_score_delta": post_metrics["weighted_score"] - baseline_metrics["weighted_score"],
}

print("\nBaseline vs fine-tuned deltas:")
for k, v in comparison.items():
    print(f"{k}: {v:+.4f}")
